# ONH Node and Link Extraction Experiments

This notebook checks the graph extraction stage after ONH binarisation and skeletonisation. It does not change the GUI, Retina code, or the production ONH pipeline.

Workflow:
1. Load the four ONH test images and fixed annular ROI.
2. Generate binary masks using the selected ONH binarisation method.
3. Generate skeletons using the selected balanced pruning strategy.
4. Extract graph nodes and links using `skel2graph3d`.
5. Visualise nodes, links, link length distributions, and optional short-link filtering.

The goal is to decide whether the current node/link extraction is acceptable or whether extra graph filtering is needed before biomarker calculation.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

from scipy.ndimage import convolve, distance_transform_edt
from skimage.morphology import medial_axis, remove_small_objects

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gui" / "build" / "modules" / "Skel2Graph3D.py").exists():
            if (candidate / "algorithms test images").exists():
                return candidate
    raise FileNotFoundError("Could not find the OCTAVA project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / "algorithms test images"
GUI_BUILD_DIR = PROJECT_ROOT / "gui" / "build"
MODULE_DIR = GUI_BUILD_DIR / "modules"

for item in (str(GUI_BUILD_DIR), str(MODULE_DIR)):
    if item not in sys.path:
        sys.path.insert(0, item)

from modules.onh_binarise_image import binarise_onh_image
from modules.Skel2Graph3D import skel2graph3d

METADATA_PATH = TEST_DIR / "test images information.xlsx"
OUTPUT_DIR = TEST_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Test image folder:", TEST_DIR)
print("Metadata:", METADATA_PATH)

## Load Test Images and Annular ROI

The annular ROI is fixed at 2-4 mm diameter within a 4.5 mm x 4.5 mm ONH scan.

In [ ]:
IMAGE_SIZE_PX = 1500
SCAN_SIZE_MM = 4.5
INNER_DIAMETER_MM = 2.0
OUTER_DIAMETER_MM = 4.0
BOUNDARY_MARGIN_PX = 8
SHORT_LINK_THRESHOLD_PX = 15

metadata = pd.read_excel(METADATA_PATH)
metadata

In [ ]:
def read_grayscale_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)


def resize_to_scan(image: np.ndarray, image_size_px: int = IMAGE_SIZE_PX) -> np.ndarray:
    interpolation = cv2.INTER_AREA if max(image.shape) > image_size_px else cv2.INTER_CUBIC
    return cv2.resize(image, (image_size_px, image_size_px), interpolation=interpolation).astype(np.float64)


def annulus_mask(
    image_shape: tuple[int, int],
    center_x: float,
    center_y: float,
    inner_diameter_mm: float = INNER_DIAMETER_MM,
    outer_diameter_mm: float = OUTER_DIAMETER_MM,
    scan_size_mm: float = SCAN_SIZE_MM,
) -> np.ndarray:
    h, w = image_shape
    px_per_mm = w / scan_size_mm
    inner_radius_px = (inner_diameter_mm / 2.0) * px_per_mm
    outer_radius_px = (outer_diameter_mm / 2.0) * px_per_mm
    yy, xx = np.ogrid[:h, :w]
    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    return (distance >= inner_radius_px) & (distance <= outer_radius_px)


def load_cases(metadata: pd.DataFrame) -> list[dict]:
    cases = []
    for _, row in metadata.iterrows():
        image_path = TEST_DIR / str(row["filename"])
        raw = read_grayscale_image(image_path)
        image = resize_to_scan(raw)
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        roi_mask = annulus_mask(image.shape, center_x, center_y)
        device = str(row["device"]).strip()
        quality = str(row["quality"]).strip()
        cases.append(
            {
                "name": f"{device} {quality}",
                "filename": str(row["filename"]),
                "path": image_path,
                "raw_shape": raw.shape,
                "image": image,
                "roi_mask": roi_mask,
                "center_x": center_x,
                "center_y": center_y,
                "device": device,
                "quality": quality,
            }
        )
    return cases


cases = load_cases(metadata)
[(case["name"], case["raw_shape"], case["image"].shape, (case["center_x"], case["center_y"])) for case in cases]

## Generate Binary and Skeleton Inputs

The skeleton is generated using the selected **balanced pruning** strategy from the skeletonisation comparison.

In [ ]:
class NotebookVar:
    def __init__(self, value):
        self.value = value
    def get(self):
        return self.value


class NotebookCanvas:
    status_id = 1
    frangi_bool = NotebookVar(1)
    frangi_min = NotebookVar(1)
    frangi_max = NotebookVar(4)
    med_filt_bool = NotebookVar(0)
    kernel_size = NotebookVar(3)

    def itemconfigure(self, *args, **kwargs):
        pass

    def update_idletasks(self):
        pass


def generate_binary(case: dict) -> np.ndarray:
    canvas = NotebookCanvas()
    canvas.onh_mask_img = case["image"].astype(np.float64)
    canvas.onh_mask_arr = case["roi_mask"].astype(bool)
    binarise_onh_image(canvas, display=False)
    return canvas.binary_image.astype(np.uint8)


def neighbour_count_8(skeleton: np.ndarray) -> np.ndarray:
    kernel = np.array(
        [[1, 1, 1],
         [1, 0, 1],
         [1, 1, 1]],
        dtype=np.uint8,
    )
    return convolve(skeleton.astype(np.uint8), kernel, mode="constant", cval=0)


def prune_short_spurs(skeleton: np.ndarray, min_branch_length: int = 12, max_iterations: int = 10) -> np.ndarray:
    skel = skeleton.astype(bool).copy()

    for _ in range(max_iterations):
        neighbour_count = neighbour_count_8(skel)
        endpoints = np.argwhere(skel & (neighbour_count == 1))
        if len(endpoints) == 0:
            break

        pixels_to_remove = []
        for endpoint in endpoints:
            path = []
            current = tuple(endpoint)
            previous = None

            for _step in range(min_branch_length + 1):
                path.append(current)
                y, x = current
                neighbours = []

                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < skel.shape[0] and 0 <= nx < skel.shape[1] and skel[ny, nx]:
                            if previous is None or (ny, nx) != previous:
                                neighbours.append((ny, nx))

                if len(neighbours) == 0:
                    break

                if len(neighbours) >= 2:
                    if len(path) <= min_branch_length:
                        pixels_to_remove.extend(path[:-1])
                    break

                previous = current
                current = neighbours[0]

        if not pixels_to_remove:
            break

        for y, x in pixels_to_remove:
            skel[y, x] = False

    return skel


def generate_balanced_skeleton(binary: np.ndarray, roi_mask: np.ndarray) -> dict:
    roi_mask = roi_mask.astype(bool)
    vessel_mask = (binary > 0) & roi_mask
    image_size = vessel_mask.shape[0]

    pre_min_area = max(20, int(image_size * image_size * 0.00001))
    vessel_mask_clean = remove_small_objects(vessel_mask, min_size=pre_min_area, connectivity=2)
    vessel_mask_clean = vessel_mask_clean & roi_mask

    skeleton_raw, medial_distance = medial_axis(vessel_mask_clean, return_distance=True)
    skeleton_raw = skeleton_raw & roi_mask
    distance_map = distance_transform_edt(vessel_mask_clean)

    skeleton_component_pruned = remove_small_objects(skeleton_raw, min_size=8, connectivity=2)
    skeleton_component_pruned = skeleton_component_pruned & roi_mask

    skeleton = prune_short_spurs(skeleton_component_pruned, min_branch_length=15, max_iterations=10)
    skeleton = skeleton & roi_mask

    return {
        "vessel_mask_clean": vessel_mask_clean,
        "skeleton_raw": skeleton_raw,
        "skeleton": skeleton,
        "distance_map": distance_map,
    }


binary_results = {}
skeleton_results = {}
for case in cases:
    binary = generate_binary(case)
    binary_results[case["name"]] = binary
    skeleton_results[case["name"]] = generate_balanced_skeleton(binary, case["roi_mask"])

print("Generated binary and balanced-pruned skeletons.")

## Extract Graph Nodes and Links

`skel2graph3d` converts the cleaned skeleton into graph nodes and links. Endpoints and branchpoints are represented as nodes, while vessel segments between nodes are represented as links.

In [ ]:
def flat_indices_to_rc(indices, shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    indices = np.asarray(indices, dtype=int)
    if indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    rows, cols = np.unravel_index(indices, shape)
    return rows, cols


def node_center_rc(node, shape: tuple[int, int]) -> tuple[float, float]:
    rows, cols = flat_indices_to_rc(getattr(node, "nodes_idx", []), shape)
    if rows.size == 0:
        return np.nan, np.nan
    return float(np.mean(rows)), float(np.mean(cols))


def link_path_rc(link, shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    return flat_indices_to_rc(getattr(link, "path", []), shape)


def is_near_annulus_boundary(row: float, col: float, case: dict, shape: tuple[int, int], margin_px: float = BOUNDARY_MARGIN_PX) -> bool:
    px_per_mm = shape[1] / SCAN_SIZE_MM
    inner_radius_px = (INNER_DIAMETER_MM / 2.0) * px_per_mm
    outer_radius_px = (OUTER_DIAMETER_MM / 2.0) * px_per_mm
    distance = np.sqrt((col - case["center_x"]) ** 2 + (row - case["center_y"]) ** 2)
    return abs(distance - inner_radius_px) <= margin_px or abs(distance - outer_radius_px) <= margin_px


def extract_graph_for_case(case: dict, skeleton: np.ndarray, twig_size: int = 8) -> dict:
    nodes, links = skel2graph3d(skeleton.astype(bool), twig_size)
    shape = skeleton.shape

    node_rows = []
    for idx, node in enumerate(nodes):
        row, col = node_center_rc(node, shape)
        is_endpoint = getattr(node, "ep", 0) == 1
        node_rows.append(
            {
                "node_index": idx,
                "row": row,
                "col": col,
                "node_type": "endpoint" if is_endpoint else "branchpoint",
                "link_count": len(getattr(node, "links", [])),
                "near_annulus_boundary": is_near_annulus_boundary(row, col, case, shape),
            }
        )

    link_rows = []
    for idx, link in enumerate(links):
        rr, cc = link_path_rc(link, shape)
        length_px = int(len(rr))
        start_node = int(getattr(link, "start", -1))
        end_node = int(getattr(link, "end", -1))
        start_type = node_rows[start_node]["node_type"] if 0 <= start_node < len(node_rows) else "unknown"
        end_type = node_rows[end_node]["node_type"] if 0 <= end_node < len(node_rows) else "unknown"
        near_boundary = False
        if 0 <= start_node < len(node_rows) and node_rows[start_node]["near_annulus_boundary"]:
            near_boundary = True
        if 0 <= end_node < len(node_rows) and node_rows[end_node]["near_annulus_boundary"]:
            near_boundary = True

        link_rows.append(
            {
                "link_index": idx,
                "start": start_node,
                "end": end_node,
                "start_type": start_type,
                "end_type": end_type,
                "length_px": length_px,
                "is_short_link": length_px <= SHORT_LINK_THRESHOLD_PX,
                "touches_boundary_node": near_boundary,
            }
        )

    return {
        "nodes": nodes,
        "links": links,
        "node_table": pd.DataFrame(node_rows),
        "link_table": pd.DataFrame(link_rows),
    }


graph_results = {}
for case in cases:
    skeleton = skeleton_results[case["name"]]["skeleton"]
    graph_results[case["name"]] = extract_graph_for_case(case, skeleton, twig_size=8)

print("Extracted graph nodes and links.")

## Graph Summary Table

This table helps detect whether the graph extraction produces too many endpoints, branchpoints, or short links.

In [ ]:
def graph_summary_table(cases: list[dict], graph_results: dict) -> pd.DataFrame:
    rows = []
    for case in cases:
        name = case["name"]
        node_table = graph_results[name]["node_table"]
        link_table = graph_results[name]["link_table"]

        endpoint_count = int((node_table["node_type"] == "endpoint").sum()) if len(node_table) else 0
        branchpoint_count = int((node_table["node_type"] == "branchpoint").sum()) if len(node_table) else 0
        boundary_endpoint_count = int(((node_table["node_type"] == "endpoint") & node_table["near_annulus_boundary"]).sum()) if len(node_table) else 0

        if len(link_table):
            link_lengths = link_table["length_px"].to_numpy()
            short_link_count = int(link_table["is_short_link"].sum())
            short_link_pct = 100 * short_link_count / len(link_table)
            boundary_link_pct = 100 * int(link_table["touches_boundary_node"].sum()) / len(link_table)
            median_link_length = float(np.median(link_lengths))
            mean_link_length = float(np.mean(link_lengths))
            p10_link_length = float(np.percentile(link_lengths, 10))
            p90_link_length = float(np.percentile(link_lengths, 90))
        else:
            short_link_count = 0
            short_link_pct = 0.0
            boundary_link_pct = 0.0
            median_link_length = 0.0
            mean_link_length = 0.0
            p10_link_length = 0.0
            p90_link_length = 0.0

        rows.append(
            {
                "case": name,
                "node_count": len(node_table),
                "endpoint_count": endpoint_count,
                "branchpoint_count": branchpoint_count,
                "boundary_endpoint_count": boundary_endpoint_count,
                "link_count": len(link_table),
                "short_link_count": short_link_count,
                "short_link_pct": short_link_pct,
                "boundary_link_pct": boundary_link_pct,
                "median_link_length_px": median_link_length,
                "mean_link_length_px": mean_link_length,
                "p10_link_length_px": p10_link_length,
                "p90_link_length_px": p90_link_length,
            }
        )
    return pd.DataFrame(rows)


graph_summary = graph_summary_table(cases, graph_results)
graph_summary

## Nodes Overlay

Yellow points are branchpoint nodes. Red points are endpoint nodes. This plot is useful for checking whether nodes are concentrated in plausible vessel junctions or whether the graph creates excessive false nodes.

In [ ]:
def normalize_for_display(image: np.ndarray, roi_mask: np.ndarray | None = None) -> np.ndarray:
    image = image.astype(np.float64)
    pixels = image.ravel() if roi_mask is None else image[roi_mask]
    if pixels.size == 0:
        return np.zeros_like(image, dtype=np.uint8)
    lo, hi = np.percentile(pixels, (1, 99))
    if hi <= lo:
        lo, hi = float(pixels.min()), float(pixels.max())
    out = (image - lo) / (hi - lo + 1e-8)
    out = np.clip(out, 0, 1)
    return (out * 255).astype(np.uint8)


def plot_nodes_overlay(cases: list[dict], skeleton_results: dict, graph_results: dict) -> None:
    fig, axes = plt.subplots(1, len(cases), figsize=(4.2 * len(cases), 4.2))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        name = case["name"]
        base = normalize_for_display(case["image"], case["roi_mask"])
        base = np.where(case["roi_mask"], base, 0)
        skeleton = skeleton_results[name]["skeleton"]
        node_table = graph_results[name]["node_table"]

        ax.imshow(base, cmap="gray", vmin=0, vmax=255)
        rr, cc = np.where(skeleton)
        ax.scatter(cc, rr, s=0.2, c="cyan", alpha=0.35, linewidths=0)

        if len(node_table):
            endpoints = node_table[node_table["node_type"] == "endpoint"]
            branchpoints = node_table[node_table["node_type"] == "branchpoint"]
            ax.scatter(branchpoints["col"], branchpoints["row"], s=10, c="yellow", alpha=0.9, linewidths=0)
            ax.scatter(endpoints["col"], endpoints["row"], s=10, c="red", alpha=0.9, linewidths=0)

        ax.set_title(name)
        ax.axis("off")
    fig.suptitle("Graph node overlay: yellow=branchpoint nodes, red=endpoint nodes", y=0.98)
    plt.tight_layout()


plot_nodes_overlay(cases, skeleton_results, graph_results)

## Links Coloured by Length

This plot checks whether a large number of very short links are present. Short links are more likely to be spurs, small noisy branches, or local over-segmentation artefacts.

In [ ]:
def link_segments_for_plot(links, shape: tuple[int, int], max_links: int | None = None):
    segments = []
    lengths = []
    selected_links = links if max_links is None else links[:max_links]
    for link in selected_links:
        rr, cc = link_path_rc(link, shape)
        if len(rr) < 2:
            continue
        points = np.column_stack([cc, rr])
        segments.append(points)
        lengths.append(len(rr))
    return segments, np.asarray(lengths, dtype=float)


def plot_links_by_length(cases: list[dict], graph_results: dict, skeleton_results: dict) -> None:
    all_lengths = []
    for case in cases:
        link_table = graph_results[case["name"]]["link_table"]
        if len(link_table):
            all_lengths.extend(link_table["length_px"].tolist())
    vmax = np.percentile(all_lengths, 95) if all_lengths else 1
    norm = Normalize(vmin=0, vmax=max(vmax, SHORT_LINK_THRESHOLD_PX + 1))

    fig, axes = plt.subplots(1, len(cases), figsize=(4.4 * len(cases), 4.2))
    if len(cases) == 1:
        axes = [axes]

    for ax, case in zip(axes, cases):
        name = case["name"]
        skeleton = skeleton_results[name]["skeleton"]
        links = graph_results[name]["links"]
        shape = skeleton.shape
        segments, lengths = link_segments_for_plot(links, shape)

        ax.imshow(np.zeros_like(skeleton), cmap="gray", vmin=0, vmax=1)
        if len(segments):
            collection = LineCollection(segments, cmap="viridis", norm=norm, linewidths=0.8)
            collection.set_array(lengths)
            ax.add_collection(collection)
        ax.set_title(name)
        ax.set_xlim(0, shape[1])
        ax.set_ylim(shape[0], 0)
        ax.axis("off")

    fig.suptitle("Graph links coloured by link length", y=0.98)
    plt.tight_layout()
    if all_lengths:
        cbar = fig.colorbar(collection, ax=axes, fraction=0.025, pad=0.01)
        cbar.set_label("Link length (pixels)")


plot_links_by_length(cases, graph_results, skeleton_results)

## Link Length Histograms

The dashed red line marks the current short-link threshold used for inspection.

In [ ]:
def plot_link_length_histograms(cases: list[dict], graph_results: dict) -> None:
    fig, axes = plt.subplots(1, len(cases), figsize=(4.0 * len(cases), 3.2))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        link_table = graph_results[case["name"]]["link_table"]
        lengths = link_table["length_px"].to_numpy() if len(link_table) else np.array([])
        ax.hist(lengths, bins=40, color="#4f80ff", edgecolor="white")
        ax.axvline(SHORT_LINK_THRESHOLD_PX, color="red", linestyle="--", linewidth=1.2)
        ax.set_title(case["name"])
        ax.set_xlabel("Link length (pixels)")
        ax.set_ylabel("Count")
    fig.suptitle("Distribution of graph link lengths", y=1.03)
    plt.tight_layout()


plot_link_length_histograms(cases, graph_results)

## Optional Short-Link Filtering View

This does not change the production code. It only visualises what the graph would look like if links shorter than the threshold were ignored during later graph-based metric calculation.

In [ ]:
def plot_original_vs_short_link_filtered(cases: list[dict], graph_results: dict, skeleton_results: dict) -> None:
    fig, axes = plt.subplots(len(cases), 2, figsize=(8.0, 3.8 * len(cases)), squeeze=False)

    for r, case in enumerate(cases):
        name = case["name"]
        skeleton = skeleton_results[name]["skeleton"]
        links = graph_results[name]["links"]
        link_table = graph_results[name]["link_table"]
        shape = skeleton.shape

        for c, title in enumerate(["Original links", f"Ignoring links <= {SHORT_LINK_THRESHOLD_PX}px"]):
            ax = axes[r, c]
            ax.imshow(np.zeros_like(skeleton), cmap="gray", vmin=0, vmax=1)

            if c == 0:
                kept_link_indices = set(range(len(links)))
            else:
                kept_link_indices = set(link_table.loc[~link_table["is_short_link"], "link_index"].astype(int).tolist())

            segments = []
            lengths = []
            for idx, link in enumerate(links):
                if idx not in kept_link_indices:
                    continue
                rr, cc = link_path_rc(link, shape)
                if len(rr) < 2:
                    continue
                segments.append(np.column_stack([cc, rr]))
                lengths.append(len(rr))

            if segments:
                collection = LineCollection(segments, colors="white", linewidths=0.8)
                ax.add_collection(collection)

            if c == 0:
                ax.set_ylabel(name, rotation=90, labelpad=8)
            ax.set_title(title if r == 0 else "")
            ax.set_xlim(0, shape[1])
            ax.set_ylim(shape[0], 0)
            ax.axis("off")

    fig.suptitle("Visual effect of ignoring very short graph links", y=0.995)
    plt.tight_layout()


plot_original_vs_short_link_filtered(cases, graph_results, skeleton_results)

## How to Interpret This Notebook

The current `skel2graph3d` output can be accepted if:

1. Branchpoint nodes are located at plausible vessel junctions.
2. Endpoint nodes are mostly caused by real vessel endings or annulus boundaries.
3. Very short links are not dominant in the graph summary table.
4. Ignoring short links does not remove a large amount of visually meaningful vasculature.

If many short links or false branchpoint clusters are visible, graph-level filtering should be considered before biomarker calculation.